# 09 — Mission constraints and physical-balance sensitivity

This notebook exposes interactions hidden by a single design-point number. It
recomputes mission geometry from YAML and cross-checks selected outputs with
simple equations. It reads current CAD balance if present; it does not create a
new approved configuration or silently resize the aircraft.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=False)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
from dataclasses import replace
acquisition_cam = replace(cam, max_fps=min(cam.max_fps, s["acquisition_rate_cap_hz"]))
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=140)
    display(Image(filename=str(path)))
    plt.close(fig)


## Wind changes ground speed independently of stall margin

For a straight commanded ground track, let $w_\parallel$ be wind along that track
(tailwind positive) and $w_\perp$ its crosswind component. At true airspeed $V_a$,
heading correction gives
$$V_g=\sqrt{V_a^2-w_\perp^2}+w_\parallel.$$
Plain-text equation: `groundspeed = sqrt(airspeed^2 - crosswind^2) + tailwind`.
A real positive root requires $|w_\perp|<V_a$ and forward progress requires $V_g>0$.
At fixed airspeed, a tailwind can violate image blur or frame-rate limits while a
headwind reduces progress. The angle below is the **wind velocity vector relative
to the track**, not a meteorological wind-from bearing. The plot uses the mission's
maximum wind and the study altitude factors; it does not set a flight wind limit.


In [ ]:
max_wind = float(d.mission["mission"]["max_wind_speed_ms"])
v_air = d.selected.cruise_speed_m_s
v_stall = d.selected.stall_speed_m_s
angles = np.linspace(0, 360, studies["plot_points"] * 4 + 1)
wind_rows = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for fraction in [0.0, 1.0]:
    wind = max_wind * fraction
    along = wind * np.cos(np.radians(angles))
    cross = wind * np.sin(np.radians(angles))
    ground = np.where(abs(cross) < v_air,
                      np.sqrt(np.maximum(0, v_air**2-cross**2)) + along, np.nan)
    axes[0].plot(angles, ground, label=f"Wind {wind:.1f} m/s")
axes[0].axhline(d.survey_speed_limit_m_s, ls="--", color="C3", label="Nominal imaging speed ceiling")
axes[0].set(xlabel="Wind vector relative to track [deg; 0 = tailwind]", ylabel="Groundspeed at design airspeed [m/s]")
for case, parallel, perpendicular in (("headwind", -max_wind, 0), ("crosswind", 0, max_wind), ("tailwind", max_wind, 0)):
    ground = np.sqrt(v_air**2-perpendicular**2) + parallel if abs(perpendicular) < v_air else np.nan
    wind_rows.append({"case": case, "airspeed_m_s": v_air, "groundspeed_m_s": ground,
                      "trigger_hz": sv.trigger_rate_hz(cam, s["agl_m"], ground, s["forward_overlap"]),
                      "motion_blur_px": sv.blur_px(cam, s["agl_m"], ground, s["exposure_s"]),
                      "within_imaging_ceiling": bool(0 < ground <= d.survey_speed_limit_m_s)})
heights = np.asarray(studies["altitude_factors"]) * s["agl_m"]
limits = np.array([min(sv.max_ground_speed_m_s(acquisition_cam, h, s["forward_overlap"]),
                      sv.max_speed_for_blur(cam, h, s["exposure_s"], s["blur_limit_px"])) for h in heights])
axes[1].plot(heights, limits-v_air, "o-", label="Tailwind ceiling at design cruise")
axes[1].plot(heights, limits-v_stall*req["launch_margin"], "s--", label="At launch-margin × stall (reference only)")
axes[1].axhline(max_wind, color="C3", ls=":", label="Mission maximum wind")
axes[1].axhline(0, color="black", lw=.7)
axes[1].set(xlabel="AGL [m]", ylabel="Allowable along-track tailwind from imaging [m/s]")
for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=.25)
save(fig, "09_wind_and_imaging.png")
display(pd.DataFrame(wind_rows))
print("The launch margin is shown only as a reference airspeed, not a qualified survey-speed control law.")


## Small changes around the selected point

At constant mass and wing, $V_s\propto\sqrt{m}$ and
$\partial x_{CG}/\partial x_{battery}=m_{battery}/m_{aircraft}$. At fixed pack and
electrical load shape, reserve allocations reduce usable survey time. The figure
below keeps the existing endurance model result and subtracts explicit mission-time
allocations; these allocations are a sensitivity sweep from zero to the configured
turn-time fraction, **not** measured transit times. The original turn efficiency is
still applied within the remaining survey window, avoiding double-counting it as
available on-line time.

Plain-text equations: `CG_shift = pack_mass / aircraft_mass * pack_position_shift`;
`survey_time = predicted_endurance * (1 - extra_time_allocation)`.
Coverage uses cruise airspeed as a calm-wind groundspeed reference; real wind
and separate climb/transit power demands require a segment-by-segment mission model.


In [ ]:
fractions = np.linspace(0, 1-s["turn_efficiency"], studies["plot_points"])
fig, ax = plt.subplots(figsize=(8, 4))
for name, ev in d.evaluations.items():
    remaining = ev.endurance_min * (1-fractions)
    coverage = [sv.coverage_ha(cam, s["agl_m"], ev.cruise_speed_m_s, s["side_overlap"], t*60, s["turn_efficiency"]) for t in remaining]
    ax.plot(fractions*100, coverage, label=f"{name}: estimated full duration {ev.endurance_min:.1f} min")
ax.set(xlabel="Additional flight time allocated outside survey [%]", ylabel="Remaining survey coverage upper bound [ha]")
ax.legend(); ax.grid(alpha=.25)
save(fig, "09_coverage_allocation.png")
manifest_path = REPO_ROOT / "outputs/parts/manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
if manifest is None:
    print("No CAD balance available: build parts before the physical-balance sensitivity review.")
else:
    balance = manifest["balance"]
    fab = yaml.safe_load((PLATFORM_DIR / "fabrication.yaml").read_text())
    total = balance["auw_g"]
    pack_mass = d.electronics.pack.mass_g
    fig, ax = plt.subplots(figsize=(9, 4))
    for bay in fab["layout"]["battery_placements"]:
        xs = np.linspace(*bay["cg_range_mm"], studies["plot_points"])
        cg = balance["cg_x_mm"] + pack_mass/total * (xs-balance["battery_x_mm"])
        sm = (balance["x_np_mm"]-cg)/manifest["design"]["mac_mm"]
        ax.plot(xs, 100*sm, label=f"{bay['name']} envelope; all other masses fixed")
    ax.axhline(100*pl["static_margin"], color="black", ls="--", label="Design target")
    ax.scatter([balance["battery_x_mm"]], [100*balance["static_margin"]], color="C3", label="Current CAD balance", zorder=5)
    ax.set(xlabel="Battery CG x aft of root LE [mm]", ylabel="Predicted static margin [% MAC]")
    ax.legend(fontsize=8); ax.grid(alpha=.25)
    save(fig, "09_battery_position_sensitivity.png")
    display(pd.DataFrame([{"aircraft_mass_g":total, "pack_mass_g":pack_mass,
                           "cg_movement_per_mm_pack_movement":pack_mass/total,
                           "target_reachable_in_current_build":balance["target_cg_reachable"]}]))
    print("Aft-bay points are moment sensitivities only. They do not prove installed fit, trim authority or an acceptable static margin.")


## Independent arithmetic and reviewer questions

The following checks use direct equations rather than calling the corresponding
survey/wing model helper. They can expose a units or stale-output mistake, but
still share the input assumptions and therefore are not independent experimental
validation. Review the wind conflict, discrete-pack endurance shortfall, and
unreachable baseline CG as design decisions; do not suppress them because an
individual model or mesh export passed.


In [ ]:
summary = d.selected.summary()
independent_area = .5*(d.wing.root_chord_mm+d.wing.tip_chord_mm)*(2*d.wing.half_span_mm)/1e6
independent_gsd = s["agl_m"] * s["camera"]["pixel_um"]*1e-6 / (s["camera"]["focal_mm"]*1e-3)
checks = [
    {"quantity":"Trapezoidal wing area [m2]", "direct_equation":independent_area, "model_result":d.selected.wing_area_m2},
    {"quantity":"Aspect ratio [-]", "direct_equation":d.selected.span_m**2/independent_area, "model_result":pl["aspect_ratio"]},
    {"quantity":"GSD [m/pixel]", "direct_equation":independent_gsd, "model_result":cam.gsd_m(s["agl_m"])},
    {"quantity":"Cruise CL [-]", "direct_equation":2*d.selected.auw_kg*G/(d.rho*d.selected.cruise_speed_m_s**2*independent_area), "model_result":summary["cl_cruise"]},
]
for row in checks:
    row["absolute_difference"] = abs(row["direct_equation"]-row["model_result"])
display(pd.DataFrame(checks))
requirements = [
    {"review_item":"Launch release / stall", "prediction":req["hand_launch_speed_m_s"]/v_stall, "bound":req["launch_margin"], "comparison":">=", "model_pass":req["hand_launch_speed_m_s"] >= req["launch_margin"]*v_stall},
    {"review_item":"Selected payload endurance [min]", "prediction":d.selected.endurance_min, "bound":req["endurance_min"], "comparison":">=", "model_pass":d.selected.endurance_min >= req["endurance_min"]},
    {"review_item":"Cruise + maximum tailwind [m/s]", "prediction":v_air+max_wind, "bound":d.survey_speed_limit_m_s, "comparison":"<=", "model_pass":v_air+max_wind <= d.survey_speed_limit_m_s},
]
display(pd.DataFrame(requirements))
print("Model pass is only the listed arithmetic comparison; no flight authorization or hardware qualification is implied.")
